## Create a Model Module for Training

In [1]:
import torch
from torch.utils.data import Dataset
from torchvision import datasets
from torchvision import transforms as tt
import matplotlib.pyplot as plt
from monai import transforms as mT  ## Breaks with numpy > 2.0
from monai.utils import set_determinism
import timm
import os
from pathlib import PosixPath, Path
import json
import numpy as np
import yaml
from typing import List, Dict, Tuple, Optional, Union, Any
from tqdm.notebook import tqdm
import nrrd
import pandas as pd
from dotenv import load_dotenv

In [2]:
os.getcwd()

'/home/vignesh/Documents/mednist_trials/notebooks/binary'

In [3]:
load_dotenv("../../envs/nvidia.env")
root_dir = Path(os.environ.get("DATASET_DIR"))
data_dir = Path(os.environ.get("DATA_DIR"))
cache_dir = Path(os.environ.get("CACHE_DIR"))
set_determinism(seed=42)

In [4]:
with open(data_dir / "hyperparam_nvidia_binary.yml", "r") as outfile:
    hparams_dict = yaml.safe_load(outfile)

In [5]:
with open(str(data_dir / "random_split.json"), "r") as fp:
    data_split = json.load(fp)

In [6]:
def replace_header(path: str, pattern: str, replace_str: str) -> str:
    return path.replace(
        pattern,
        replace_str,
    )


## Preprocessing
for split_type in ["train", "ftune", "test"]:
    data_split[split_type]["image"] = [
        replace_header(
            path=img_path, pattern="<DATASET_DIR>", replace_str=str(root_dir)
        )
        for img_path in data_split[split_type]["image"]
    ]

In [7]:
breast_split = {}
for split_type in ["train", "ftune", "test"]:
    breast_split[split_type] = {"image": [], "label": []}
    for idx, dirname in enumerate(data_split[split_type]["image"]):
        if "BreastMRI" in dirname:
            breast_split[split_type]['image'].append(dirname)
            breast_split[split_type]['label'].append("1")
        else:
            breast_split[split_type]['image'].append(dirname)
            breast_split[split_type]['label'].append("0")


In [8]:
## Define all relevant transforms!
train_transforms = mT.Compose(
    [
        mT.LoadImage(image_only=True),
        mT.EnsureChannelFirst(),  ## Add a channel to the batch dimension
        mT.ScaleIntensity(),
        mT.RandRotate(range_x=np.pi / 12, prob=0.5, keep_size=True),
        mT.RandFlip(spatial_axis=0, prob=0.5),
        mT.RandZoom(min_zoom=0.9, max_zoom=1.1, prob=0.5),
        mT.ToTensor(),
    ]
)

ftune_transforms = mT.Compose(
    [
        mT.LoadImage(image_only=True),
        mT.EnsureChannelFirst(),  ## Add a channel to the batch dimension
        mT.ScaleIntensity(),
    ]
)

pred_transform = mT.Compose([mT.Activations(sigmoid=True)])

label_transform = mT.Compose(mT.AsDiscrete(to_onehot=hparams_dict["out_channels"]))

In [9]:
trainDict_transforms = mT.Compose(
    [
        mT.LoadImaged(keys=["image"], image_only=True),
        mT.EnsureChannelFirstd(keys=["image"]),  ## Add a channel to the batch dimension
        mT.ScaleIntensityd(keys=["image"]),
        mT.RandRotated(keys=["image"], range_x=np.pi / 12, prob=0.5, keep_size=True),
        mT.RandFlipd(keys=["image"], spatial_axis=0, prob=0.5),
        mT.RandZoomd(keys=["image"], min_zoom=0.9, max_zoom=1.1, prob=0.5),
        mT.ToTensord(
            keys=["image", "label"],
        ),
    ]
)
ftuneDict_transforms = mT.Compose(
    [
        mT.LoadImaged(keys=["image"], image_only=True),
        mT.EnsureChannelFirstd(keys=["image"]),  ## Add a channel to the batch dimension
        mT.ScaleIntensityd(keys=["image"]),
        mT.ToTensord(
            keys=["image", "label"],
        ),
    ]
)

In [11]:
## Dataset!
class MedNIST_Dataset(torch.utils.data.Dataset):
    def __init__(
        self,
        data_dict: Dict,
        transforms: mT.Compose,
        image_key: str = "image",
        label_key: str = "label",
    ) -> None:
        self.data = data_dict
        self.transform = transforms
        self.image_key = image_key
        self.label_key = label_key

    def __len__(self):
        return len(self.data[self.image_key])

    def __getitem__(self, index):
        return {
            "x": self.transform(self.data[self.image_key][index]),
            "y": int(self.data[self.label_key][index]),
        }

In [17]:
train_ds = MedNIST_Dataset(
    data_dict=breast_split["train"],
    transforms=train_transforms,
)

ftune_ds = MedNIST_Dataset(
    data_dict=breast_split["ftune"],
    transforms=ftune_transforms,
)

## Dataloaders!
train_dl = torch.utils.data.DataLoader(train_ds, batch_size=1024, num_workers=8)

ftune_dl = torch.utils.data.DataLoader(ftune_ds, batch_size=1024, num_workers=8)

In [18]:
from monai.data import PersistentDataset

trainDs = PersistentDataset(
    data=breast_split["train"],
    transform=trainDict_transforms,
    cache_dir=cache_dir,
)
ftuneDs = PersistentDataset(
    data=breast_split["ftune"],
    transform=ftuneDict_transforms,
    cache_dir=cache_dir,
)

## Dataloaders!
trainDL = torch.utils.data.DataLoader(trainDs, batch_size=1024, num_workers=8)

ftuneDL = torch.utils.data.DataLoader(ftuneDs, batch_size=1024, num_workers=8)

## Model Def

In [24]:
class_weights

array([39986,  7177])

In [37]:
from monai.networks import nets as monai_nets

torch_device = torch.device(hparams_dict["torch_device"])
net = timm.create_model(
    "resnet34",
    pretrained=hparams_dict["pretrained"],
    in_chans=hparams_dict["in_channels"],
    num_classes=hparams_dict["out_channels"],
).to(torch_device)

if hparams_dict["torch_device"] == "cuda":
    net = torch.compile(net)

class_counts = np.unique(breast_split['train']['label'], return_counts=True)[1]
class_weights = torch.tensor(max(class_counts) / len(breast_split['train']['image'])).to(torch_device)

## Training related:
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=class_weights)
optimizer = torch.optim.AdamW(
    net.parameters(), 
    lr=float(hparams_dict["lr"]),
    weight_decay=float(hparams_dict['wd']))

In [38]:
train_batch = next(iter(train_dl))

In [49]:
train_batch['y'].shape

torch.Size([1024])

In [57]:
from monai.metrics import ROCAUCMetric
from torchmetrics import F1Score, Accuracy, AUROC

rocauc = ROCAUCMetric()
## Set average to None to get classwise.
acc = Accuracy(task="binary", num_classes=hparams_dict["out_channels"])
f1 = F1Score(task="binary", num_classes=hparams_dict["out_channels"])
auroc = AUROC(task="binary", num_classes=hparams_dict["out_channels"])

with torch.no_grad():
    outs = net(train_batch["x"].to(torch_device))
    labels = torch.nn.functional.one_hot(
        train_batch['y'].to(int), num_classes=hparams_dict["out_channels"]).float().to(torch_device)

    loss = criterion(outs, labels).cpu()
    pred = torch.stack([pred_transform(out.cpu()) for out in outs])
    
    y_pred = pred  ## Append or cat with multi-batch
    y_gt = labels

# acc = torch.eq(torch.stack(y_pred).argmax(dim=1), train_batch['y']).astype(int).mean() # Channel dimension is 1
out_acc = acc(y_pred.argmax(dim=1).cpu(), y_gt.argmax(dim=1).cpu())
out_f1 = f1(y_pred.argmax(dim=1).cpu(), y_gt.argmax(dim=1).cpu())
out_auroc = auroc(y_pred.cpu(), y_gt.cpu())

# metric = rocauc(y_pred, y_gt)
# metric = rocauc.aggregate()

In [67]:
from torchmetrics import Accuracy, AUROC

metric_suite = {
    "auroc": AUROC(task="binary", num_classes=hparams_dict["out_channels"]),
    "acc": Accuracy(task="binary", num_classes=hparams_dict["out_channels"]),
    "f1": F1Score(task="binary", num_classes=hparams_dict["out_channels"]),
}
metric_suite

{'auroc': BinaryAUROC(), 'acc': BinaryAccuracy(), 'f1': BinaryF1Score()}

In [16]:
hparams_dict["out_channels"]

6

In [77]:
def train_epoch(
    net: Any,
    train_dl: torch.utils.data.DataLoader,
    torch_device: str,
    log_tracker: Dict,
    metric_suite: Dict,
):
    net.train()
    epoch_loss, epoch_acc, epoch_f1, epoch_auroc, step = 0, 0, 0, 0, 0

    for batch in tqdm(train_dl):

        ## Transfer to device
        imgs = batch["x"].to(torch_device)
        labels = torch.nn.functional.one_hot(
            batch['y'].to(int), 
            num_classes=hparams_dict["out_channels"]).float().to(torch_device)
        
        optimizer.zero_grad(set_to_none=True) ## Reset Optimizer

        ## Output
        outputs = net(imgs)

        ## Compute loss and back-prop
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        with torch.no_grad():
            act = torch.sigmoid(outputs)
            epoch_acc += metric_suite['acc'](act.argmax(dim=1).cpu(),  batch['y'].cpu())
            epoch_f1 += metric_suite['f1'](act.argmax(dim=1).cpu(),  batch['y'].cpu())
            epoch_auroc += metric_suite['auroc'](act.cpu(), labels.to(torch.long).cpu())
            epoch_loss += loss.cpu()

        epoch_loss += loss.item()
        step += 1

    ## Average over steps
    epoch_loss /= step
    epoch_f1 /= step
    epoch_acc /= step
    epoch_auroc /= step

    log_tracker['train_loss'].append(epoch_loss)
    log_tracker['train_f1'].append(epoch_f1)
    log_tracker['train_acc'].append(epoch_acc)
    log_tracker['train_auroc'].append(epoch_auroc)

    del imgs, labels, batch, outputs
    return log_tracker

In [78]:
def val_epoch(
    net: Any,
    val_dl: torch.utils.data.DataLoader,
    torch_device: str,
    log_tracker: Dict,
    metric_suite: Dict,
    split_type: str = "ftune",
):

    net.eval()
    epoch_loss, epoch_acc, epoch_f1, epoch_auroc, step = 0, 0, 0, 0, 0

    with torch.no_grad():
        for batch in tqdm(val_dl):
            ## Transfer to device
            imgs = batch["x"].to(torch_device)
            labels = torch.nn.functional.one_hot(
                batch['y'].to(int), 
                num_classes=hparams_dict["out_channels"]).float().to(torch_device)
        
            optimizer.zero_grad(set_to_none=True) ## Reset Optimizer

            ## Output
            outputs = net(imgs)

            ## Compute loss and back-prop
            loss = criterion(outputs, labels)
            
            ## Compute metrics.
            act = torch.sigmoid(outputs)
            epoch_acc += metric_suite['acc'](act.argmax(dim=1).cpu(),  batch['y'].cpu())
            epoch_f1 += metric_suite['f1'](act.argmax(dim=1).cpu(),  batch['y'].cpu())
            epoch_auroc += metric_suite['auroc'](act.cpu(), labels.to(torch.long).cpu())
            epoch_loss += loss.item()
            step += 1

    ## Average over steps
    epoch_loss /= step
    epoch_f1 /= step
    epoch_acc /= step
    epoch_auroc /= step

    log_tracker['ftune_loss'].append(epoch_loss)
    log_tracker['ftune_f1'].append(epoch_f1)
    log_tracker['ftune_acc'].append(epoch_acc)
    log_tracker['ftune_auroc'].append(epoch_auroc)

    del imgs, labels, batch, outputs
    return log_tracker

In [79]:
log_tracker = {}
split_type = "ftune"
for key in ["loss", "acc", "auroc", "f1"]:
    log_tracker[f"{split_type}_{key}"] = []
split_type = "train"
for key in ["loss", "acc", "auroc", "f1"]:
    log_tracker[f"{split_type}_{key}"] = []

# log_tracker = val_epoch(
#         net=net,
#         val_dl=ftune_dl,
#         torch_device=torch_device,
#         log_tracker=log_tracker,
#         split_type="ftune",)

In [80]:
log_tracker = train_epoch(
        net=net,
        train_dl=train_dl,
        torch_device=torch_device,
        log_tracker=log_tracker,
        metric_suite=metric_suite,
    )

  0%|          | 0/47 [00:00<?, ?it/s]

In [81]:
log_tracker =  val_epoch(
    net=net,
    val_dl=ftune_dl,
    torch_device=torch_device,
    log_tracker=log_tracker,
    metric_suite=metric_suite)

  0%|          | 0/6 [00:00<?, ?it/s]

In [82]:
log_tracker

{'ftune_loss': [0.01709689696629842],
 'ftune_acc': [metatensor(0.9998)],
 'ftune_auroc': [metatensor(1.0000)],
 'ftune_f1': [metatensor(0.9995)],
 'train_loss': [metatensor(0.0554)],
 'train_acc': [metatensor(0.9999)],
 'train_auroc': [metatensor(1.0000)],
 'train_f1': [metatensor(0.9997)]}

In [23]:
## Training:

for epoch in tqdm(range(hparams_dict["epochs"])):
    ## train:
    log_tracker = train_epoch(
        net=net,
        train_dl=train_dl,
        torch_device=torch_device,
        log_tracker=log_tracker,
    )
    ## validate/finetune
    log_tracker = val_epoch(
        net=net,
        val_dl=ftune_dl,
        torch_device=torch_device,
        log_tracker=log_tracker,
        split_type="ftune",
    )

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/47 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

(metatensor(0.8901), metatensor(0.5676), metatensor(0.5676))


  0%|          | 0/47 [00:00<?, ?it/s]

...

  0%|          | 0/6 [00:00<?, ?it/s]

(metatensor(0.9718), metatensor(0.7852), metatensor(0.7852))


  0%|          | 0/47 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

(metatensor(0.9837), metatensor(0.8495), metatensor(0.8495))
